## 汎用プロンプトテンプレート


In [23]:
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("./.env.local")

# zero-shot
prompt_template = PromptTemplate.from_template(
    "私の隣人の苗字は{lastname}です。{gender}が生まれたので、名前を考えてください。簡潔に答えてください。"
)
model = ChatOpenAI(
    model="gemini-2.5-flash",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    streaming=True,
)

# .formatメソッドで情報を注入すればOK
# prompt_text = prompt_template.format(lastname="张", gender="女儿")
#
# model = Tongyi(model="qwen-max")
# res = model.invoke(input=prompt_text)
# print(res)

chain = prompt_template | model

res = chain.invoke(input={"lastname": "张", "gender": "女儿"})
print(res.content)

おめでとうございます！いくつか候補を挙げますね。

*   **張 雅 (Zhāng Yǎ)**：優雅、上品
*   **張 欣 (Zhāng Xīn)**：喜び、幸福
*   **張 慧 (Zhāng Huì)**：聡明、賢い
*   **張 静怡 (Zhāng Jìngyí)**：穏やかで楽しい
*   **張 夢瑶 (Zhāng Mèngyáo)**：夢のように美しい玉

ご夫婦で素敵な名前を選んでくださいね。


## FewShotプロンプトテンプレート

In [8]:
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("./.env.local")
example_template = PromptTemplate.from_template("単語：{word}, 反意語：{antonym}")
model = ChatOpenAI(
    model="gemini-2.5-flash",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    streaming=True,
)
examples_data = [
    {"word": "大", "antonym": "小"},
    {"word": "上", "antonym": "下"},
]

few_shot_template = FewShotPromptTemplate(
    example_prompt=example_template,
    examples=examples_data,
    prefix="単語の反意語を教えてください。以下の例を示します：",
    suffix="前述の例を参考に、{input_word}の反意語は何ですか？",
    input_variables=["input_word"], )

prompt_text = few_shot_template.invoke(input={"input_word": "左"}).to_string()
print(prompt_text)

print(model.invoke(input=prompt_text).content)

単語の反意語を教えてください。以下の例を示します：

単語：大, 反意語：小

単語：上, 反意語：下

前述の例を参考に、左の反意語は何ですか？
単語：左, 反意語：右


## ChatPromptTemplateの使い方

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("./.env.local")
chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个边塞诗人，可以作诗。"),
        MessagesPlaceholder("history"),
        ("human", "请再来一首唐诗"),
    ]
)
model = ChatOpenAI(
    model="gemini-2.5-flash",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    streaming=True,
)
history_data = [
    ("human", "你来写一个唐诗"),
    ("ai", "床前明月光，疑是地上霜，举头望明月，低头思故乡"),
    ("human", "好诗再来一个"),
    ("ai", "锄禾日当午，汗滴禾下锄，谁知盘中餐，粒粒皆辛苦"),
]

prompt_text = chat_prompt_template.invoke({"history": history_data}).to_string()

print(model.invoke(prompt_text).content)

好的，再来一首：

春眠不觉晓，
处处闻啼鸟。
夜来风雨声，
花落知多少。
